# Session 6 - inpainting, token budget, crop control


**GPU T4 x2, ~3 h.** Three additional conditions, each run on a small subset under its
own tag so that Session 11 reports them as separate rows:

* **inpainting comparison** -- the counterfactual is produced by inpainting region k
  instead of splicing the authentic pixels, and the two operators are compared;
* **visual-token budget** -- whether faithfulness changes as the encoder sees fewer tokens;
* **crop control** -- whether removing background context changes the verdict.

Run this only after Session 5 has reached its sample target for this model.

In [ ]:
SESSION = "S6 extras"

# ============================== CONFIG ==============================
MODEL        = "qwen25vl:Qwen/Qwen2.5-VL-3B-Instruct"
QUANT        = "fp16"
MAX_PIXELS   = 384 * 384
SEED         = 0
SPLIT        = "test"

INPAINT_METHODS = ["telea", "lama"]   # [] = skip; lama installs a package
INPAINT_SAMPLES = 300

ABL_MAX_PIXELS  = [256 * 256]         # [] = skip
ABL_SAMPLES     = 150

CROP_CONTROL_INDEX = ""   # index.json from a CROP_SIZE=0 run of Session 1
CROP_SAMPLES       = 200

EXTRA_BUDGET_MIN = 150    # per extra

In [ ]:
# ---------------------------------------------------------------- BOOT
# Locates the code dataset wherever it is mounted, puts it on sys.path,
# prints the attached inputs, and records provenance.  The search is by
# file name, so the dataset's mount name does not matter.
import os, sys, subprocess, json, time

def _find_code():
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if os.path.basename(dirpath) == "ccaudit" and "kaggle_utils.py" in files:
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not find the ccaudit package.\n"
        "Add Input -> your code dataset (<your-code-dataset>), and check that "
        "the preview shows ccaudit/kaggle_utils.py at the top level.")

CODE = _find_code()
if CODE not in sys.path:
    sys.path.insert(0, CODE)
# Child processes do not inherit sys.path.  Every `python -m ccaudit.<module>`
# below runs as a subprocess, so the code directory must be on PYTHONPATH.
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
from ccaudit import kaggle_utils as KU
from ccaudit import common as C

OUT = KU.work_dir("audit")
TMP = KU.temp_dir()
os.environ["HF_HOME"] = KU.temp_dir("hf")          # model weights stay out of /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KU.session_header(SESSION, OUT)
print("code:", CODE)

In [ ]:
# ------------------------------------------------------- SELF TEST (always)
# The self-test suite runs in under a minute and needs no dataset.  Each check
# corresponds to a failure mode that would produce plausible-looking but
# incorrect numbers, so a failure here invalidates everything that follows.
rc = KU.sh(f"{sys.executable} {CODE}/scripts/selftest.py", check=False)
if rc != 0:
    raise SystemExit("SELF TEST FAILED -- inspect the failures above before proceeding.")

In [ ]:
KU.pip_install("transformers accelerate qwen-vl-utils")
if "lama" in INPAINT_METHODS:
    KU.pip_install("simple-lama-inpainting")
KU.gpu_report()
INDEX = KU.find_parsed_index()
recs, meta = C.load_index(INDEX)
VOCAB = meta.get("vocab", "face8")
RESUME = ",".join(d for d in KU.find_run_dirs() if "run_" in d)
n_gpu = max(1, KU.n_gpus())
print("INDEX =", INDEX, "| GPUs:", n_gpu)

def fan_out(tag, flags, samples, budget, out_sub="run_extras", index=None):
    cmds, envs, logs = [], [], []
    for i in range(n_gpu):
        logs.append(f"{OUT}/logs/{tag}_{i}.log")
        envs.append({"CUDA_VISIBLE_DEVICES": str(i)})
        cmds.append(
            f'{sys.executable} -m ccaudit.m5_runner '
            f'--index "{index or INDEX}" --detector "{MODEL}" '
            f'--out "{OUT}/{out_sub}" --tag {tag} --split {SPLIT} '
            f'--limit-samples {samples} --seed {SEED} --shard {i}/{n_gpu} '
            f'--device cuda --quant {QUANT} --max-pixels {MAX_PIXELS} '
            f'--vocab {VOCAB} --time-budget-min {budget} {flags}'
            + (f' --resume-from "{RESUME}"' if RESUME else ""))
    print(C.banner(tag))
    KU.run_parallel(cmds, envs=envs, logs=logs, check=False, poll_sec=60)

In [ ]:
# ==================== INPAINTING COMPARISON ====================
for m in INPAINT_METHODS:
    fan_out(f"inpaint_{m}", f"--inpaint {m} --splice-floor",
            INPAINT_SAMPLES, EXTRA_BUDGET_MIN)

In [ ]:
# ==================== VISUAL-TOKEN BUDGET ====================
# The runner appends ":px<max_pixels>" to the cache key only when MAX_PIXELS
# differs from the default (384*384), so a non-default budget never reuses the
# default run's predictions.  Each budget still carries its own tag so that
# its rows are reported separately.
for px in ABL_MAX_PIXELS:
    cmds, envs, logs = [], [], []
    for i in range(n_gpu):
        logs.append(f"{OUT}/logs/px{px}_{i}.log")
        envs.append({"CUDA_VISIBLE_DEVICES": str(i)})
        cmds.append(
            f'{sys.executable} -m ccaudit.m5_runner --index "{INDEX}" '
            f'--detector "{MODEL}" --out "{OUT}/run_extras" --tag px{px} '
            f'--split {SPLIT} --limit-samples {ABL_SAMPLES} --seed {SEED} '
            f'--shard {i}/{n_gpu} --device cuda --quant {QUANT} '
            f'--max-pixels {px} --vocab {VOCAB} '
            f'--time-budget-min {EXTRA_BUDGET_MIN}')
    print(C.banner(f"token budget {px}"))
    KU.run_parallel(cmds, envs=envs, logs=logs, check=False, poll_sec=60)

In [ ]:
# ==================== CROP CONTROL ====================
# Requires a second Session 1 run with CROP_SIZE = 0 on a subset.
if CROP_CONTROL_INDEX:
    fan_out("fullframe", "", CROP_SAMPLES, EXTRA_BUDGET_MIN,
            index=CROP_CONTROL_INDEX)
else:
    print("crop control skipped. To run it: re-run Session 1 with "
          "CROP_SIZE = 0 and MAX_PAIRS = 200, make a dataset, attach it here "
          "and set CROP_CONTROL_INDEX to its parsed/index.json.")

In [ ]:
KU.sh(f'{sys.executable} -m ccaudit.m6_metrics --raw "{OUT}/run_extras" '
      f'--out "{OUT}/metrics_extras" --split {SPLIT} --vocab {VOCAB}',
      check=False)
for r in C.load_json(f"{OUT}/metrics_extras/metrics.json", {}).get("results", []):
    ip = r.get("inpaint") or {}
    print(f"{r['detector']} [{r.get('tag')}]: FS={r.get('FS'):.4f}"
          + (f"  | inpaint: corr={ip.get('corr_inpaint_vs_delta'):.3f} "
             f"sign-disagree={ip.get('sign_disagreement'):.3f} "
             f"p-rise-on-REAL={ip.get('p_rise_on_real'):.4f} "
             f"manufactures_evidence={ip.get('manufactures_evidence')}"
             if ip else ""))

In [ ]:
NEXT_STEP = """1. Output tab -> New Dataset -> `cca-s6-extras`.
2. In the inpainting table, **p rise on REAL** measures the inpainter's effect
   on authentic frames. A confidence interval that excludes zero indicates
   that the inpainting operator itself introduces forgery evidence, which
   disqualifies it as a counterfactual operator; this is recorded as a
   property of the operator."""

# ----------------------------------------------------------- WRAP UP
KU.disk_report()
print(C.banner("NEXT STEP"))
print(NEXT_STEP)